In [1]:
!pip install transformers>=4.40.0 datasets evaluate sacrebleu \
            torch torchvision accelerate sentencepiece sqlglot -q

#Train Config

In [2]:
train_config = {
    # Model
    'model_name': 't5-base',
    'src_pfx'   : '<sqlite>',
    'tgt_pfx'   : '<hive>',
    'max_ip_ln': 256,
    'max_op_ln': 256,
    # DATA
    'train_file': '/content/DSAI_project/split/train.csv',
    'valid_file': '/content/DSAI_project/split/val.csv',
    'test_file' : '/content/DSAI_project/split/test.csv',
    # Training
    'num_epochs'       : 10,
    'batch_size'       : 16,                 # reduce to 8 if GPU OOM
    'grad_accum_steps' : 2,                  # effective batch = 32
    'learning_rate'    : 3e-4,
    'warmup_ratio'     : 0.06,
    'weight_decay'     : 0.01,
    'lr_scheduler'     : 'cosine',
    'fp16'             : True,
    # Generation
    'num_beams'        : 4,
    'early_stopping'   : True,
    'max_new_tokens'   : 256,
    #Checkpointing
    'ckpt_dir'         : 'checkpoints',
    'save_total_lmt'      : 3,
    'metric_for_best'  : 'eval_exact_match',
    # Reproducibility
    'seed'             : 42


}

In [3]:
import random, os
import numpy as np
import torch


In [4]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(train_config['seed'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {device}")
print(f"Model  : {train_config['model_name']}")
print(f"Epochs : {train_config['num_epochs']}  |  Batch : {train_config['batch_size']}  |  LR : {train_config['learning_rate']}")

Device : cuda
Model  : t5-base
Epochs : 10  |  Batch : 16  |  LR : 0.0003


In [6]:
import pandas as pd
from datasets import Dataset

In [7]:
train_df = pd.read_csv(train_config['train_file'])
val_df   = pd.read_csv(train_config['valid_file'])
test_df  = pd.read_csv(train_config['test_file'])


In [8]:

print("Split sizes:")
print(f"  train : {len(train_df):,} rows")
print(f"  val   : {len(val_df):,} rows")
print(f"  test  : {len(test_df):,} rows")
print()
print("Complexity distribution (train):")
print(train_df['complexity'].value_counts().to_string())
print()
print("Synthetic vs original (train):")
print(train_df['is_synthetic'].value_counts().to_string())

# ── Convert to HuggingFace Dataset ────────────────────────────────────────
train_ds = Dataset.from_pandas(train_df[['sqlite', 'hive']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df  [['sqlite', 'hive']].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df [['sqlite', 'hive']].reset_index(drop=True))

print("\nDataset objects created.")
print(train_ds)

Split sizes:
  train : 3,064 rows
  val   : 963 rows
  test  : 888 rows

Complexity distribution (train):
complexity
High Complexity      1607
Medium Complexity     844
Low Complexity        613

Synthetic vs original (train):
is_synthetic
True     2671
False     393

Dataset objects created.
Dataset({
    features: ['sqlite', 'hive'],
    num_rows: 3064
})


#Tokenization

In [9]:
from transformers import T5Tokenizer

In [10]:
tokenizer = T5Tokenizer.from_pretrained(train_config['model_name'])
num_added = tokenizer.add_tokens(['<sqlite>', '<hiveql>', '<sep>'], special_tokens=True)
print(f"Special tokens added : {num_added}")
print(f"Vocabulary size      : {len(tokenizer):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens added : 3
Vocabulary size      : 32,103


In [11]:
def preprocess(batch):
  inputs  = [train_config['src_pfx'] + q for q in batch['sqlite']]
  targets = [train_config['tgt_pfx'] + q for q in batch['hive']]

  # Tokenise source sequences
  model_inputs = tokenizer(
        inputs,
        max_length  = train_config['max_ip_ln'],
        padding     = 'max_length',
        truncation  = True,
    )
  # Tokenise target sequences
  # The as_target_tokenizer context manager is deprecated.
  # We can directly tokenize the targets.
  labels = tokenizer(
      targets,
      max_length  = train_config['max_op_ln'],
      padding     = 'max_length',
      truncation  = True,
  )
  model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
  return model_inputs

In [12]:
tokenised_train = train_ds.map(preprocess, batched=True, remove_columns=['sqlite', 'hive'])
tokenised_val   = val_ds.map(preprocess,   batched=True, remove_columns=['sqlite', 'hive'])
tokenised_test  = test_ds.map(preprocess,  batched=True, remove_columns=['sqlite', 'hive'])

print("\nTokenised splits:")
print(f"  train : {len(tokenised_train):,}")
print(f"  val   : {len(tokenised_val):,}")
print(f"  test  : {len(tokenised_test):,}")

Map:   0%|          | 0/3064 [00:00<?, ? examples/s]

Map:   0%|          | 0/963 [00:00<?, ? examples/s]

Map:   0%|          | 0/888 [00:00<?, ? examples/s]


Tokenised splits:
  train : 3,064
  val   : 963
  test  : 888


In [13]:

sample_src = train_config['src_pfx'] + train_df['sqlite'].iloc[0]
sample_tgt = train_config['tgt_pfx'] + train_df['hive'].iloc[0]
print(f"\nSample input  : {sample_src[:120]}")
print(f"Sample target : {sample_tgt[:120]}")
print(f"Input tokens  : {tokenizer(sample_src, return_length=True)['length'][0]}")
print(f"Target tokens : {tokenizer(sample_tgt, return_length=True)['length'][0]}")


Sample input  : <sqlite>SELECT DISTINCT COUNT( PAPERalias0.PAPERID ) FROM AUTHOR AS AUTHORalias0 , PAPER AS PAPERalias0 , WRITES AS WRIT
Sample target : <hive>SELECT DISTINCT COUNT(PAPERalias0.PAPERID) FROM AUTHOR AS AUTHORalias0, PAPER AS PAPERalias0, WRITES AS WRITESalia
Input tokens  : 149
Target tokens : 149


#Model initialization

In [14]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(train_config['model_name'])

# Resize token embeddings to cover the 3 newly added special tokens
model.resize_token_embeddings(len(tokenizer))

model = model.to(device)

# ── Parameter summary ─────────────────────────────────────────────────────
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"Embedding matrix     : {model.shared.weight.shape}  (vocab × d_model)")
print(f"Model on device      : {next(model.parameters()).device}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total parameters     : 222,884,352
Trainable parameters : 222,884,352
Embedding matrix     : torch.Size([32103, 768])  (vocab × d_model)
Model on device      : cuda:0


#Evaluate Metrcs

In [15]:
import evaluate
import sqlglot

bleu_metric = evaluate.load('sacrebleu')

In [16]:
def normalise_sql(sql: str) -> str:
    return ' '.join(sql.lower().split())

def is_valid_hive(sql: str) -> bool:
    try:
        sqlglot.parse_one(sql, read='hive')
        return True
    except Exception:
        return False

def compute_metrics(predictions: list, references: list) -> dict:
    norm_preds = [normalise_sql(p) for p in predictions]
    norm_refs  = [normalise_sql(r) for r in references]

    exact_match  = sum(p == r for p, r in zip(norm_preds, norm_refs)) / max(len(norm_preds), 1)
    bleu         = bleu_metric.compute(
                       predictions=norm_preds,
                       references=[[r] for r in norm_refs])['score']
    parse_valid  = sum(is_valid_hive(p) for p in predictions) / max(len(predictions), 1)

    return {
        'exact_match'    : round(exact_match * 100, 2),   # %
        'bleu'           : round(bleu, 2),
        'parse_validity' : round(parse_valid * 100, 2),   # %
    }


In [17]:
def compute_metrics_trainer(eval_pred):
    preds, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return compute_metrics(decoded_preds, decoded_labels)

print("Metric functions defined: compute_metrics(), compute_metrics_trainer()")

Metric functions defined: compute_metrics(), compute_metrics_trainer()


#Training Loop

In [18]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import os

os.makedirs(train_config['ckpt_dir'], exist_ok=True)


In [19]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model               = model,
    label_pad_token_id  = -100,
    pad_to_multiple_of  = 8,    # efficient on tensor cores
)

In [20]:
data_collator

DataCollatorForSeq2Seq(tokenizer=T5Tokenizer(name_or_path='t5-base', vocab_size=32100, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, ls

In [21]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = train_config['ckpt_dir'],
    num_train_epochs            = train_config['num_epochs'],
    per_device_train_batch_size = train_config['batch_size'],
    per_device_eval_batch_size  = train_config['batch_size'],
    gradient_accumulation_steps = train_config['grad_accum_steps'],
    learning_rate               = train_config['learning_rate'],
    warmup_ratio                = train_config['warmup_ratio'],
    weight_decay                = train_config['weight_decay'],
    lr_scheduler_type           = train_config['lr_scheduler'],
    fp16                        = train_config['fp16'],
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = train_config['metric_for_best'],
    greater_is_better           = True,
    save_total_limit            = train_config['save_total_lmt'],
    predict_with_generate       = True,
    generation_num_beams        = train_config['num_beams'],
    generation_max_length       = train_config['max_new_tokens'],
    logging_dir                 = './logs',
    logging_steps               = 50,
    seed                        = train_config['seed'],
    report_to                   = 'none',  # swap to 'wandb' for experiment tracking
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenised_train,
    eval_dataset    = tokenised_val,
    data_collator   = data_collator,
    tokenizer       = tokenizer,
    compute_metrics = compute_metrics_trainer,
)

TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'

In [23]:
# ── Trainer ────────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenised_train,
    eval_dataset    = tokenised_val,
    data_collator   = data_collator,
    compute_metrics = compute_metrics_trainer,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# ── Train ──────────────────────────────────────────────────────────────────
trainer.train()

Epoch,Training Loss,Validation Loss


OverflowError: out of range integral type conversion attempted

#test Set Evaluation

In [1]:
# ── Predict on test set ────────────────────────────────────────────────────
test_results = trainer.predict(tokenised_test)

pred_ids  = test_results.predictions
label_ids = test_results.label_ids


NameError: name 'trainer' is not defined